# Modelado de la demanda MiBici — resultados de la Fase 2

**TC5044 Operaciones de aprendizaje automático · Fase 2 · Equipo 15**

Este notebook **no contiene lógica del pipeline**: importa los módulos refactorizados del paquete
`tc5044_mlops` y solo sirve para explorar y comunicar resultados. El entrenamiento se ejecuta
desde la terminal (`make train` o `mlflow run . -e train --env-manager local`) para que sea
reproducible; aquí se leen las corridas de MLflow y el Model Registry.

| Paso | Módulo |
|---|---|
| Parámetros (rutas, cortes, hiperparámetros, semillas) | `params.yaml` → `config.cargar_params` |
| Tabla de modelado estación × hora | `features.py` |
| Preprocesamiento + modelo (`Pipeline` de scikit-learn) | `pipeline.py` |
| Catálogo de modelos y justificación | `modeling/models.py` |
| Métricas y línea base | `modeling/evaluate.py` |
| Búsqueda de hiperparámetros (Optuna) | `modeling/tune.py` |
| Entrenamiento, MLflow Tracking y Model Registry | `modeling/train.py`, `tracking.py` |
| Inferencia con el modelo `@champion` | `modeling/predict.py` |

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import Image, display
import mlflow
from mlflow import MlflowClient
import pandas as pd
from sklearn import set_config

from tc5044_mlops.config import FIGURES_DIR, cargar_params
from tc5044_mlops.features import cargar_tabla
from tc5044_mlops.modeling.models import JUSTIFICACION, crear_modelo
from tc5044_mlops.modeling.predict import cargar_modelo, predecir
from tc5044_mlops.modeling.train import preparar_datos
from tc5044_mlops.pipeline import construir_pipeline
from tc5044_mlops.tracking import configurar_mlflow

pd.set_option("display.width", 140)
params = cargar_params()

## 1. Datos y partición cronológica

La tabla la genera la etapa `features` (`python -m tc5044_mlops.features`). La partición es por fecha
(cortes en `params.yaml`), nunca aleatoria.

In [ ]:
tabla = cargar_tabla()
datos = preparar_datos(tabla, params)
pd.DataFrame({
    "conjunto": ["train", "val", "test"],
    "filas": [len(datos["X_tr"]), len(datos["X_va"]), len(datos["X_te"])],
    "demanda_media": [datos["y_tr"].mean(), datos["y_va"].mean(), datos["y_te"].mean()],
    "pct_ceros": [100 * (datos[k] == 0).mean() for k in ("y_tr", "y_va", "y_te")],
}).round(3)

## 2. El pipeline de scikit-learn

Un único `Pipeline` encadena el preprocesamiento y el modelo, así las transformaciones se ajustan
solo con train y se aplican igual en validación, prueba y producción:

- **numéricas**: imputación por mediana (rezagos de la primera semana) + `StandardScaler`;
- **estación**: `TargetEncoder` (demanda media suavizada, con *cross fitting*) + escalado;
- **perfil horario**: `PerfilHorarioPCA`, PCA sobre la proporción de demanda por hora de cada estación
  (laborable y fin de semana), que resume su tipo de uso.

In [ ]:
set_config(display="diagram")
construir_pipeline(crear_modelo("hgb"))

## 3. Modelos candidatos y justificación

In [ ]:
pd.DataFrame(JUSTIFICACION.items(), columns=["modelo", "justificación"])

## 4. Experimentos registrados en MLflow

Cada ejecución de `make train` crea una corrida padre con una corrida hija por modelo y una nieta
por intento de Optuna. Se selecciona por `val_mae`; `test_*` solo estima el desempeño fuera de muestra.

In [ ]:
configurar_mlflow(params)
corridas = mlflow.search_runs(experiment_names=[params["mlflow"]["experimento"]])
padre = corridas[corridas["tags.tipo"] == "corrida_padre"].sort_values("start_time").iloc[-1]
hijas = corridas[(corridas["tags.mlflow.parentRunId"] == padre["run_id"])
                 & corridas["metrics.val_mae"].notna()]
cols = ["tags.modelo", "metrics.val_mae", "metrics.val_rmse", "metrics.val_wape",
        "metrics.val_r2", "metrics.test_mae", "metrics.tiempo_entrenamiento_seg"]
(hijas[cols].rename(columns=lambda c: c.split(".", 1)[1])
     .sort_values("val_mae").reset_index(drop=True).round(4))

In [ ]:
display(Image(FIGURES_DIR / "fase2_comparacion_modelos.png"))
display(Image(FIGURES_DIR / "fase2_optuna_historial.png"))

## 5. Model Registry: versiones, alias y métricas

In [ ]:
cliente = MlflowClient()
nombre = params["mlflow"]["modelo_registrado"]
alias = {v: a for a, v in cliente.get_registered_model(nombre).aliases.items()}
pd.DataFrame([{
    "version": int(v.version),
    "alias": alias.get(v.version, ""),
    "modelo": v.tags.get("modelo"),
    "val_mae": float(v.tags.get("val_mae")),
    "test_mae": float(v.tags.get("test_mae")),
    "run_id": v.run_id[:8],
} for v in cliente.search_model_versions(f"name='{nombre}'")]).sort_values("version", ascending=False)

## 6. Modelo campeón: desempeño en prueba y tipos de estación

In [ ]:
for archivo in ("fase2_campeon_serie_test.png", "fase2_campeon_real_vs_predicho_test.png",
                "fase2_perfiles_pca_estaciones.png"):
    display(Image(FIGURES_DIR / archivo))

## 7. Inferencia con el alias `@champion`

`predict.py` carga el modelo por alias, no por número de versión: si un reentrenamiento promueve
otro modelo, el código de inferencia no cambia.

In [ ]:
modelo = cargar_modelo("champion", params)
muestra = tabla[tabla["fecha_hora"] >= params["particion"]["corte_test"]].tail(24)
predecir(muestra, modelo, params).round(2)